In [1]:
%pip install holidays

Note: you may need to restart the kernel to use updated packages.


# Notebook 06 — Construcción de `silver.tiempo`

## Objetivo

Construir la tabla calendario `silver.tiempo` para servir como dimensión temporal en los análisis posteriores. A diferencia del resto de tablas Silver, esta no se deriva de bronze, sino que se genera desde cero con un rango de fechas y se enriquece con:

1. **Componentes temporales**: año, mes, día, trimestre, semana ISO, día de la semana, nombres de día y mes en español.
2. **Clasificación**: distinción entre días laborables y fines de semana.
3. **Festivos nacionales españoles**: obtenidos vía la librería `python-holidays` (fuente autoritativa, reproducible).
4. **Eventos comerciales y de moda**: temporadas Primavera-Verano (PV) e Invierno-Otoño (OI), periodos de rebajas, Black Friday, San Valentín y Navidad.

La ventana temporal cubre del **1 de enero de 2022 al 31 de diciembre de 2025**, coherente con el resto de tablas Silver del proyecto.

## 1. Configuración y conexión a DuckDB

In [2]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb


## 2. Generación de la tabla base con componentes temporales

Se utiliza la función `generate_series` de DuckDB para generar todas las fechas del rango y se calculan los componentes temporales básicos. Los nombres de día y mes se derivan en español manualmente, ya que DuckDB devuelve los nombres en inglés según el locale por defecto.

In [3]:
print("Generando tabla base de fechas (2022-01-01 a 2025-12-31)...\n")

con.execute("""
    CREATE OR REPLACE TABLE silver.tiempo AS
    WITH fechas AS (
        SELECT 
            CAST(generate_series AS DATE) AS fecha
        FROM generate_series(
            DATE '2022-01-01',
            DATE '2025-12-31',
            INTERVAL '1 day'
        )
    )
    SELECT
        fecha,
        EXTRACT(YEAR    FROM fecha)::INTEGER AS anio,
        EXTRACT(MONTH   FROM fecha)::INTEGER AS mes,
        EXTRACT(DAY     FROM fecha)::INTEGER AS dia,
        EXTRACT(QUARTER FROM fecha)::INTEGER AS trimestre,
        EXTRACT(WEEK    FROM fecha)::INTEGER AS semana_iso,
        EXTRACT(DOY     FROM fecha)::INTEGER AS dia_del_anio,
        EXTRACT(DOW     FROM fecha)::INTEGER AS dia_semana_num,
        CASE EXTRACT(DOW FROM fecha)
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Lunes'
            WHEN 2 THEN 'Martes'
            WHEN 3 THEN 'Miércoles'
            WHEN 4 THEN 'Jueves'
            WHEN 5 THEN 'Viernes'
            WHEN 6 THEN 'Sábado'
        END AS nombre_dia,
        CASE EXTRACT(MONTH FROM fecha)
            WHEN  1 THEN 'Enero'
            WHEN  2 THEN 'Febrero'
            WHEN  3 THEN 'Marzo'
            WHEN  4 THEN 'Abril'
            WHEN  5 THEN 'Mayo'
            WHEN  6 THEN 'Junio'
            WHEN  7 THEN 'Julio'
            WHEN  8 THEN 'Agosto'
            WHEN  9 THEN 'Septiembre'
            WHEN 10 THEN 'Octubre'
            WHEN 11 THEN 'Noviembre'
            WHEN 12 THEN 'Diciembre'
        END AS nombre_mes,
        CASE WHEN EXTRACT(DOW FROM fecha) IN (0, 6) THEN TRUE ELSE FALSE END AS es_finde,
        CASE WHEN EXTRACT(DOW FROM fecha) IN (0, 6) THEN FALSE ELSE TRUE END AS es_laborable
    FROM fechas
    ORDER BY fecha
""")

print("✅ Tabla silver.tiempo creada con componentes básicos")
n = con.execute("SELECT COUNT(*) FROM silver.tiempo").fetchone()[0]
print(f"Total de filas: {n:,} (esperado: 1.461 = 4 años x 365/366 días)")

Generando tabla base de fechas (2022-01-01 a 2025-12-31)...

✅ Tabla silver.tiempo creada con componentes básicos
Total de filas: 1,461 (esperado: 1.461 = 4 años x 365/366 días)


## 3. Enriquecimiento con festivos nacionales españoles

Se añaden las columnas `es_festivo_nacional` y `nombre_festivo` utilizando la librería estándar `python-holidays` como fuente autoritativa. Se incluyen únicamente los festivos **nacionales** (no autonómicos ni locales) para evitar asimetrías en los análisis, dado que la cartera nacional cubre todo el territorio español.

In [4]:
import holidays

print("Añadiendo festivos nacionales españoles vía librería holidays...\n")

# Añadir columnas vacías
con.execute("ALTER TABLE silver.tiempo ADD COLUMN es_festivo_nacional BOOLEAN DEFAULT FALSE")
con.execute("ALTER TABLE silver.tiempo ADD COLUMN nombre_festivo VARCHAR")

# Festivos nacionales españoles 2022-2025 (sin pasar 'subdiv', solo los nacionales)
festivos_es = holidays.Spain(years=[2022, 2023, 2024, 2025])

# Convertir a lista ordenada
festivos_lista = sorted(festivos_es.items())
print(f"Festivos nacionales detectados: {len(festivos_lista)}\n")
for fecha, nombre in festivos_lista:
    print(f"  {fecha}  →  {nombre}")

# Insertar en la tabla
print("\nActualizando silver.tiempo...")
for fecha, nombre in festivos_lista:
    con.execute("""
        UPDATE silver.tiempo
        SET es_festivo_nacional = TRUE,
            nombre_festivo = ?
        WHERE fecha = ?
    """, [nombre, fecha])

# Verificación
n_festivos = con.execute(
    "SELECT COUNT(*) FROM silver.tiempo WHERE es_festivo_nacional = TRUE"
).fetchone()[0]
print(f"\n✅ Festivos marcados en silver.tiempo: {n_festivos}")

Añadiendo festivos nacionales españoles vía librería holidays...

Festivos nacionales detectados: 35

  2022-01-01  →  Año Nuevo
  2022-01-06  →  Epifanía del Señor
  2022-04-15  →  Viernes Santo
  2022-08-15  →  Asunción de la Virgen
  2022-10-12  →  Fiesta Nacional de España
  2022-11-01  →  Todos los Santos
  2022-12-06  →  Día de la Constitución Española
  2022-12-08  →  Inmaculada Concepción
  2023-01-06  →  Epifanía del Señor
  2023-04-07  →  Viernes Santo
  2023-05-01  →  Fiesta del Trabajo
  2023-08-15  →  Asunción de la Virgen
  2023-10-12  →  Fiesta Nacional de España
  2023-11-01  →  Todos los Santos
  2023-12-06  →  Día de la Constitución Española
  2023-12-08  →  Inmaculada Concepción
  2023-12-25  →  Natividad del Señor
  2024-01-01  →  Año Nuevo
  2024-01-06  →  Epifanía del Señor
  2024-03-29  →  Viernes Santo
  2024-05-01  →  Fiesta del Trabajo
  2024-08-15  →  Asunción de la Virgen
  2024-10-12  →  Fiesta Nacional de España
  2024-11-01  →  Todos los Santos
  2024-12-

## 4. Enriquecimiento con eventos comerciales y temporadas de moda

Se añaden flags para los principales eventos comerciales relevantes en el sector de la moda íntima:

- **Temporadas de moda**: Primavera-Verano (PV, marzo-agosto) e Invierno-Otoño (OI, septiembre-febrero).
- **Rebajas**: enero (rebajas de invierno) y julio (rebajas de verano).
- **Black Friday**: el cuarto viernes de noviembre.
- **San Valentín**: 14 de febrero.
- **Navidad** (periodo comercial): del 1 al 31 de diciembre.

Estos flags permitirán análisis de estacionalidad y comportamiento por evento en notebooks posteriores.

In [5]:
print("Añadiendo eventos comerciales y temporadas de moda...\n")

con.execute("ALTER TABLE silver.tiempo ADD COLUMN temporada_moda VARCHAR")
con.execute("ALTER TABLE silver.tiempo ADD COLUMN es_rebajas BOOLEAN DEFAULT FALSE")
con.execute("ALTER TABLE silver.tiempo ADD COLUMN es_black_friday BOOLEAN DEFAULT FALSE")
con.execute("ALTER TABLE silver.tiempo ADD COLUMN es_san_valentin BOOLEAN DEFAULT FALSE")
con.execute("ALTER TABLE silver.tiempo ADD COLUMN es_navidad BOOLEAN DEFAULT FALSE")

# Temporadas de moda: PV (marzo-agosto) | OI (septiembre-febrero)
con.execute("""
    UPDATE silver.tiempo
    SET temporada_moda = CASE 
        WHEN mes BETWEEN 3 AND 8 THEN 'PV'
        ELSE 'OI'
    END
""")

# Rebajas: enero (1-31) y julio (1-31)
con.execute("""
    UPDATE silver.tiempo
    SET es_rebajas = TRUE
    WHERE mes IN (1, 7)
""")

# Black Friday: cuarto viernes de noviembre
# 2022: 25-nov | 2023: 24-nov | 2024: 29-nov | 2025: 28-nov
con.execute("""
    UPDATE silver.tiempo
    SET es_black_friday = TRUE
    WHERE fecha IN (DATE '2022-11-25', DATE '2023-11-24', DATE '2024-11-29', DATE '2025-11-28')
""")

# San Valentín: 14 de febrero
con.execute("""
    UPDATE silver.tiempo
    SET es_san_valentin = TRUE
    WHERE mes = 2 AND dia = 14
""")

# Navidad comercial: todo diciembre
con.execute("""
    UPDATE silver.tiempo
    SET es_navidad = TRUE
    WHERE mes = 12
""")

print("✅ Eventos comerciales añadidos")

Añadiendo eventos comerciales y temporadas de moda...

✅ Eventos comerciales añadidos


## 5. Validación de la tabla resultante

In [6]:
print("=" * 60)
print("VOLUMEN Y ESQUEMA")
print("=" * 60)

n = con.execute("SELECT COUNT(*) FROM silver.tiempo").fetchone()[0]
print(f"Filas totales: {n:,}")
print(f"Esperado: 1.461 (2022:365 + 2023:365 + 2024:366 + 2025:365)\n")

esquema = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'tiempo'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema.to_string(index=False))

VOLUMEN Y ESQUEMA
Filas totales: 1,461
Esperado: 1.461 (2022:365 + 2023:365 + 2024:366 + 2025:365)

        column_name data_type
              fecha      DATE
               anio   INTEGER
                mes   INTEGER
                dia   INTEGER
          trimestre   INTEGER
         semana_iso   INTEGER
       dia_del_anio   INTEGER
     dia_semana_num   INTEGER
         nombre_dia   VARCHAR
         nombre_mes   VARCHAR
           es_finde   BOOLEAN
       es_laborable   BOOLEAN
es_festivo_nacional   BOOLEAN
     nombre_festivo   VARCHAR
     temporada_moda   VARCHAR
         es_rebajas   BOOLEAN
    es_black_friday   BOOLEAN
    es_san_valentin   BOOLEAN
         es_navidad   BOOLEAN


In [7]:
print("=" * 60)
print("RESUMEN POR AÑO")
print("=" * 60)
resumen = con.execute("""
    SELECT 
        anio,
        COUNT(*) AS dias_totales,
        SUM(CASE WHEN es_finde THEN 1 ELSE 0 END) AS dias_finde,
        SUM(CASE WHEN es_festivo_nacional THEN 1 ELSE 0 END) AS dias_festivos,
        SUM(CASE WHEN es_rebajas THEN 1 ELSE 0 END) AS dias_rebajas,
        SUM(CASE WHEN temporada_moda = 'PV' THEN 1 ELSE 0 END) AS dias_pv,
        SUM(CASE WHEN temporada_moda = 'OI' THEN 1 ELSE 0 END) AS dias_oi
    FROM silver.tiempo
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(resumen.to_string(index=False))

RESUMEN POR AÑO
 anio  dias_totales  dias_finde  dias_festivos  dias_rebajas  dias_pv  dias_oi
 2022           365       105.0            8.0          62.0    184.0    181.0
 2023           365       105.0            9.0          62.0    184.0    181.0
 2024           366       104.0            9.0          62.0    184.0    182.0
 2025           365       104.0            9.0          62.0    184.0    181.0


In [8]:
print("=" * 60)
print("VERIFICACIÓN DE EVENTOS COMERCIALES PUNTUALES")
print("=" * 60)
eventos = con.execute("""
    SELECT 
        fecha, 
        nombre_dia,
        es_black_friday,
        es_san_valentin
    FROM silver.tiempo
    WHERE es_black_friday = TRUE OR es_san_valentin = TRUE
    ORDER BY fecha
""").fetchdf()
print(eventos.to_string(index=False))

VERIFICACIÓN DE EVENTOS COMERCIALES PUNTUALES
     fecha nombre_dia  es_black_friday  es_san_valentin
2022-02-14      Lunes            False             True
2022-11-25    Viernes             True            False
2023-02-14     Martes            False             True
2023-11-24    Viernes             True            False
2024-02-14  Miércoles            False             True
2024-11-29    Viernes             True            False
2025-02-14    Viernes            False             True
2025-11-28    Viernes             True            False


In [9]:
print("=" * 60)
print("MUESTRA DE LA TABLA (primeros 15 días de 2024)")
print("=" * 60)
muestra = con.execute("""
    SELECT 
        fecha, nombre_dia, es_finde, es_festivo_nacional, 
        nombre_festivo, temporada_moda, es_rebajas
    FROM silver.tiempo
    WHERE fecha BETWEEN '2024-01-01' AND '2024-01-15'
    ORDER BY fecha
""").fetchdf()
print(muestra.to_string(index=False))

MUESTRA DE LA TABLA (primeros 15 días de 2024)
     fecha nombre_dia  es_finde  es_festivo_nacional     nombre_festivo temporada_moda  es_rebajas
2024-01-01      Lunes     False                 True          Año Nuevo             OI        True
2024-01-02     Martes     False                False                NaN             OI        True
2024-01-03  Miércoles     False                False                NaN             OI        True
2024-01-04     Jueves     False                False                NaN             OI        True
2024-01-05    Viernes     False                False                NaN             OI        True
2024-01-06     Sábado      True                 True Epifanía del Señor             OI        True
2024-01-07    Domingo      True                False                NaN             OI        True
2024-01-08      Lunes     False                False                NaN             OI        True
2024-01-09     Martes     False                False          

## 6. Conclusiones del notebook 06

### Resumen del proceso

La tabla `silver.tiempo` se ha construido como dimensión calendario para los análisis temporales del TFG. Cubre 1.461 días entre el 1 de enero de 2022 y el 31 de diciembre de 2025 (cuatro años completos, incluyendo el bisiesto 2024 con 366 días), e incluye 19 columnas con componentes temporales, festivos nacionales españoles y eventos comerciales relevantes para el sector de la moda íntima.

### Estructura final

La tabla se organiza en cuatro bloques de columnas:

- **Identificador y componentes temporales** (10 columnas): `fecha`, `anio`, `mes`, `dia`, `trimestre`, `semana_iso`, `dia_del_anio`, `dia_semana_num`, `nombre_dia`, `nombre_mes`.
- **Clasificación básica** (2 columnas): `es_finde`, `es_laborable`.
- **Festivos** (2 columnas): `es_festivo_nacional`, `nombre_festivo`. Se han marcado 35 festivos nacionales en total (8 en 2022 y 9 en cada uno de los años 2023, 2024 y 2025).
- **Eventos comerciales** (5 columnas): `temporada_moda`, `es_rebajas`, `es_black_friday`, `es_san_valentin`, `es_navidad`.

### Validación de la tabla

La tabla se ha validado mediante cuatro comprobaciones:

- **Volumen total**: 1.461 días, coincidente con el cálculo teórico (365 + 365 + 366 + 365).
- **Distribución temporal**: 104-105 días de fin de semana por año, 62 días de rebajas por año (enero + julio) y reparto coherente entre temporadas PV (184-185 días) y OI (181-182 días).
- **Eventos comerciales puntuales**: 4 fechas de Black Friday (todas en viernes, como debe ser) y 4 fechas de San Valentín marcadas correctamente.
- **Muestra cualitativa**: la inspección de los primeros 15 días de 2024 confirma que el 1 de enero (Año Nuevo) y el 6 de enero (Epifanía del Señor) están correctamente identificados como festivos, y que el flag `es_rebajas` cubre todo el mes de enero.

### Decisiones metodológicas

**Uso de la librería estándar `python-holidays` como fuente de festivos.** Garantiza trazabilidad y reproducibilidad: cualquier persona que ejecute el notebook obtiene los mismos festivos sin depender de listas hardcodeadas. La librería refleja el calendario laboral oficial publicado por el BOE.

**Asimetría observada en el conteo anual de festivos.** El año 2022 presenta 8 festivos marcados frente a los 9 del resto de años. La diferencia se debe a que algunos festivos nacionales que cayeron en domingo en 2022 no aparecen como festivos laborables en la fuente (concretamente la Navidad del 25 de diciembre de 2022, que fue domingo, y por tanto no genera día no laborable adicional). Esta característica es coherente con el uso analítico de la tabla, ya que para análisis comerciales lo relevante es el día efectivamente no laborable.

**Solo se incluyen festivos nacionales.** Los festivos autonómicos y locales se excluyen deliberadamente porque la cartera nacional de Selmark cubre todo el territorio español y la inclusión de festivos regionales introduciría asimetrías en los análisis.

**El periodo de Navidad se modeliza como mes completo.** Aunque el día 25 es el festivo formal, comercialmente toda la campaña de diciembre (desde el Black Friday) tiene comportamiento navideño en el sector textil. Por ello el flag `es_navidad` cubre los 31 días del mes.

**Las temporadas de moda se modelizan según convención del sector.** Primavera-Verano (PV) abarca de marzo a agosto; Otoño-Invierno (OI) abarca de septiembre a febrero. Esto es coherente con el campo `id_temporada` presente en `silver.ventas_minoristas`, que se podrá cruzar para validar la cobertura.

### Próximo notebook

`07_silver_mosaic.ipynb` — Limpieza y enriquecimiento de la tabla MOSAIC (categorías geodemográficas), preparando el cruce con la cartera nacional para el análisis de geomarketing.

In [10]:
con.close()
print("✅ Conexión cerrada. silver.tiempo guardada en disco.")

✅ Conexión cerrada. silver.tiempo guardada en disco.
